# 06 — Base vs Fine-Tuned Comparison

Phase 11. Same images, same prompts, same decoding. The only variable is
the LoRA adapter.


In [ ]:
# --- Colab setup (skip if running locally) ---
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('casting-defect-vlm'):
        # Replace with your repository URL, or upload the folder to Colab.
        raise SystemExit('Upload the casting-defect-vlm project folder to Colab first.')
    %cd casting-defect-vlm
    !pip install -q -r requirements.txt

sys.path.insert(0, os.path.abspath('..' if os.path.basename(os.getcwd())=='notebooks' else '.'))
print('python', sys.version.split()[0], '| colab:', IN_COLAB)


## 1. Run the comparison

This aborts if the two runs do not cover identical (image, prompt) pairs.


In [ ]:
!python scripts/compare_models.py


## 2. Quantitative table


In [ ]:
import pandas as pd, json
from pathlib import Path
comp = pd.read_csv('results/comparison/comparison.csv')
comp[comp['scope']=='overall'][['metric','base_model','finetuned_model','delta']]


## 3. Per-prompt breakdown


In [ ]:
comp[comp['scope']!='overall'].pivot_table(
    index='metric', columns='scope', values=['base_model','finetuned_model'])


## 4. Comparison plot


In [ ]:
from IPython.display import Image, display
display(Image('results/comparison/comparison_plot.png'))


## 5. The four agreement quadrants


In [ ]:
qual = pd.read_csv('results/comparison/qualitative_comparison.csv')
print(qual['quadrant'].value_counts())


### Base wrong → fine-tuned correct (the value added)


In [ ]:
fixed = qual[qual['quadrant']=='fixed_by_finetuning']
for _, r in fixed.head(3).iterrows():
    print('='*70)
    print(r['relpath'], '| truth:', r['ground_truth'], '| prompt:', r['prompt_id'])
    print('--- BASE ---'); print(r['base_response'])
    print('--- FINE-TUNED ---'); print(r['finetuned_response'])


### Base correct → fine-tuned wrong (regressions — discuss, do not hide)


In [ ]:
broke = qual[qual['quadrant']=='broken_by_finetuning']
print('regressions:', len(broke))
for _, r in broke.head(3).iterrows():
    print('='*70)
    print(r['relpath'], '| truth:', r['ground_truth'])
    print('--- BASE ---'); print(r['base_response'])
    print('--- FINE-TUNED ---'); print(r['finetuned_response'])


### Both wrong (hard cases fine-tuning did not fix)


In [ ]:
both_wrong = qual[qual['quadrant']=='both_wrong']
both_wrong[['relpath','ground_truth','base_prediction','finetuned_prediction']].head(10)


## 6. Output-format compliance and hallucination


In [ ]:
base = pd.read_csv('results/baseline/baseline_results.csv')
ft   = pd.read_csv('results/evaluation/finetuned_results.csv')
for name, df in [('base', base), ('fine-tuned', ft)]:
    unp = (df['parsed_prediction']=='Unparseable').mean()
    fmt = df['format_ok'].mean()
    halluc = ((df['ground_truth']=='OK') & (~df['predicted_defect_type'].isin(['None','Unknown']))).sum()
    print(f'{name:<12} unparseable={unp:.3f}  format_ok={fmt:.3f}  hallucinated_types={halluc}')


## 7. Confidence behaviour

Model-reported confidence is **not** a calibrated probability.


In [ ]:
for name, df in [('base', base), ('fine-tuned', ft)]:
    ok = df[df['correct']]['confidence'].mean()
    bad = df[~df['correct']]['confidence'].mean()
    print(f'{name:<12} mean confidence when correct={ok:.1f}  when wrong={bad:.1f}')


## 8. Interpretation

_Write the conclusions the numbers actually support._

> **Caveat:** any improvement here is improvement at detecting *synthetic*
> defect textures. It is not evidence of improved detection of real
> casting defects.
